# Integrated HCE Framework for Cell Type Classification

This notebook demonstrates proper integration with the HCE framework, including:
- Vectorized HCE loss computation
- Early stopping and model checkpointing
- Hierarchical accuracy metrics
- Comprehensive validation and error handling
- Reproducibility through configuration serialization

## 1. Environment Setup and Configuration

In [1]:
# Import dependencies
import os
import json
import logging
import numpy as np
import pandas as pd
from datetime import datetime

# Scientific computing
import scanpy as sc
import scipy.sparse

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# Deep learning
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# HuggingFace transformers
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

# HCE framework integration
from cell2sentence.hce_trainer import build_reachability_matrix_from_ontology

# Visualization
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

/home/hugolab/Documents/Cell2Sentence-HugoLab/Cell2Sentence-HugoLab/c2s-justin/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Configuration
CONFIG = {
    'data_path': '/home/hugolab/Documents/Cell2Sentence-HugoLab/Cell2Sentence-HugoLab/lung.h5ad',
    'output_dir': '/home/hugolab/Documents/Cell2Sentence-HugoLab/Cell2Sentence-HugoLab/lung_hce_integrated_results',
    'model_name': 'vandijklab/C2S-Pythia-410m-human-immune',
    'batch_size': 8,  # Increased for faster training
    'learning_rate': 1e-4,
    'num_epochs': 10,
    'early_stopping_patience': 3,
    'filter_unknown': True,
    'random_seed': 42,
    'min_cells_per_type': 10,  # Reduced minimum threshold
    'max_cells_per_type': 300,  # Reduced max to speed up dataset loading
    'num_genes_text': 30,  # Reduced from 50 to speed up text processing
    'max_token_length': 128,  # Reduced from 256 for faster tokenization
    'train_val_test_split': [0.7, 0.15, 0.15],
    'use_mixed_precision': True,  # Enable automatic mixed precision for memory savings
}

# Set random seeds for reproducibility
np.random.seed(CONFIG['random_seed'])
torch.manual_seed(CONFIG['random_seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG['random_seed'])

# Device setup with memory optimization
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f"Using device: {device}")

# Enable memory-efficient GPU operations
if device.type == 'cuda':
    torch.cuda.empty_cache()
    logger.info(f"GPU Memory Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Create output directory
os.makedirs(CONFIG['output_dir'], exist_ok=True)

# Save configuration for reproducibility
with open(os.path.join(CONFIG['output_dir'], 'config.json'), 'w') as f:
    json.dump(CONFIG, f, indent=2, default=str)
logger.info(f"Configuration saved to {os.path.join(CONFIG['output_dir'], 'config.json')}")

2026-03-03 14:30:34,097 - INFO - Using device: cuda
2026-03-03 14:30:34,124 - INFO - GPU Memory Available: 16.6 GB
2026-03-03 14:30:34,125 - INFO - Configuration saved to /home/hugolab/Documents/Cell2Sentence-HugoLab/Cell2Sentence-HugoLab/lung_hce_integrated_results/config.json


## 2. Load and Balance Dataset

In [ ]:
# Load dataset with memory optimization
logger.info("Loading dataset from path: %s", CONFIG['data_path'])
import gc
gc.collect()  # Clear memory before loading

adata = sc.read_h5ad(CONFIG['data_path'])
logger.info(f"Loaded dataset shape: {adata.shape}")
logger.info(f"Cell types: {adata.obs['cell_type'].nunique()}")

# Filter Unknown cells if configured - vectorized for speed
if CONFIG['filter_unknown']:
    cell_type_col = 'cell_type'
    before_filter = len(adata.obs)
    # More efficient: convert once and check
    cell_types_str = adata.obs[cell_type_col].astype(str).values
    mask = ~np.isin(cell_types_str, ['Unknown', 'unknown', 'UNKNOWN', 'nan', 'NA', 'NaN'])
    adata = adata[mask].copy()
    filtered_count = before_filter - len(adata.obs)
    logger.info(f"🔍 Filtered {filtered_count} Unknown/ambiguous cells. Remaining: {len(adata.obs)}")

# Store original counts
original_counts = adata.obs['cell_type'].value_counts()
logger.info(f"Original cell type distribution:\n{original_counts}")

# Balance dataset - vectorized approach for speed
min_cells = CONFIG['min_cells_per_type']
max_cells = CONFIG['max_cells_per_type']

# Get cell type groups efficiently
cell_types = adata.obs['cell_type'].values
unique_types = np.unique(cell_types)
balanced_indices = []

for cell_type in unique_types:
    type_indices = np.where(cell_types == cell_type)[0]
    
    # Skip if too few cells
    if len(type_indices) < min_cells:
        logger.warning(f"Skipping cell type '{cell_type}': {len(type_indices)} cells < {min_cells}")
        continue
    
    # Subsample if too many cells
    if len(type_indices) > max_cells:
        type_indices = np.random.choice(type_indices, size=max_cells, replace=False)
    
    balanced_indices.extend(type_indices)

balanced_indices = np.array(sorted(balanced_indices))
adata = adata[balanced_indices].copy()
logger.info(f"Balanced dataset shape: {adata.shape}")

balanced_counts = adata.obs['cell_type'].value_counts()
logger.info(f"Balanced cell type distribution:\n{balanced_counts}")

2026-03-03 14:30:34,136 - INFO - Loading dataset from path: /home/hugolab/Documents/Cell2Sentence-HugoLab/Cell2Sentence-HugoLab/lung.h5ad


## 3. Construct and Validate Ontology

In [ ]:
# Build ontology from cell types present in balanced dataset
unique_cell_types = sorted(adata.obs['cell_type'].unique())
logger.info(f"Building ontology for {len(unique_cell_types)} cell types")

# Create simple ontology: all types at root level (can be extended with parent relationships)
ontology = {}
for cell_type in unique_cell_types:
    ontology[cell_type] = {'parent': None}

logger.info(f"Ontology structure:\n{json.dumps(ontology, indent=2)}")

# Encode labels
le = LabelEncoder()
labels_encoded = le.fit_transform(adata.obs['cell_type'])
adata.obs['label'] = labels_encoded
num_classes = len(le.classes_)
logger.info(f"Encoded {num_classes} classes: {le.classes_}")

# Validate ontology
conflicting_ancestors = set()
for parent_type in ontology:
    if ontology[parent_type]['parent'] is not None:
        child_type = ontology[parent_type]['parent']
        if child_type in ontology and ontology[child_type].get('parent') == parent_type:
            conflicting_ancestors.add((parent_type, child_type))

if conflicting_ancestors:
    logger.warning(f"Found {len(conflicting_ancestors)} conflicting ancestor pairs: {conflicting_ancestors}")
else:
    logger.info("Ontology validation passed - no cycles detected")

## 4. Convert Gene Expression to Text

In [ ]:
# Convert gene expressions to text (top N expressed genes) - memory efficient
logger.info("Converting gene expressions to text representation")

num_genes = CONFIG['num_genes_text']

# Keep sparse format as long as possible to save memory
if scipy.sparse.issparse(adata.X):
    logger.info("Using sparse matrix format to save memory")
    cell_texts = []
    for i in range(adata.n_obs):
        # Get top expressed genes for this cell (from sparse matrix)
        row = adata.X[i].toarray().flatten()  # Convert only this row to dense
        top_genes_idx = np.argsort(-row)[:num_genes]
        top_genes = [adata.var_names[idx] for idx in top_genes_idx]
        
        # Create text representation
        text = ", ".join(top_genes)
        cell_texts.append(text)
        
        # Periodic garbage collection to prevent memory buildup
        if (i + 1) % 500 == 0:
            import gc
            gc.collect()
            if device.type == 'cuda':
                torch.cuda.empty_cache()
            logger.info(f"  Processed {i + 1}/{adata.n_obs} cells")
else:
    logger.info("Converting dense matrix to text")
    cell_texts = []
    for i in range(adata.n_obs):
        expr = adata.X[i]
        top_genes_idx = np.argsort(-expr)[:num_genes]
        top_genes = [adata.var_names[idx] for idx in top_genes_idx]
        text = ", ".join(top_genes)
        cell_texts.append(text)
        
        if (i + 1) % 500 == 0:
            import gc
            gc.collect()

adata.obs['cell_text'] = cell_texts
logger.info(f"Sample cell text (first 3):")
for i in range(min(3, len(cell_texts))):
    logger.info(f"  Cell {i}: {cell_texts[i][:100]}...")

## 5. Create Train/Validation/Test Splits

In [ ]:
# Stratified train/val/test split
logger.info("Creating stratified train/val/test splits")

train_size = CONFIG['train_val_test_split'][0]
val_size = CONFIG['train_val_test_split'][1]

# First split: train + temp (val + test)
train_idx, temp_idx = train_test_split(
    np.arange(len(adata)),
    test_size=1-train_size,
    stratify=labels_encoded,
    random_state=CONFIG['random_seed']
)

# Second split: val and test from temp
test_size_of_temp = (1 - train_size - val_size) / (1 - train_size)
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=test_size_of_temp,
    stratify=labels_encoded[temp_idx],
    random_state=CONFIG['random_seed']
)

logger.info(f"Split sizes: train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}")
logger.info(f"Proportions: train={len(train_idx)/len(adata):.2%}, val={len(val_idx)/len(adata):.2%}, test={len(test_idx)/len(adata):.2%}")

# Create split column
adata.obs['split'] = 'train'
adata.obs.loc[adata.obs_names[val_idx], 'split'] = 'val'
adata.obs.loc[adata.obs_names[test_idx], 'split'] = 'test'

## 6. Tokenization Pipeline

In [ ]:
# Load tokenizer
logger.info(f"Loading tokenizer from {CONFIG['model_name']}")
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
logger.info(f"Tokenizer vocab size: {len(tokenizer)}")

# Tokenize all texts
logger.info("Tokenizing cell texts")
tokenized = tokenizer(
    adata.obs['cell_text'].tolist(),
    max_length=CONFIG['max_token_length'],
    padding='max_length',
    truncation=True,
    return_tensors='pt'
)

logger.info(f"Tokenized shape: {tokenized['input_ids'].shape}")
logger.info(f"Sample token sequence length: {tokenized['input_ids'][0].sum(dim=0)}")

# Store for dataset creation
all_input_ids = tokenized['input_ids']
all_attention_masks = tokenized['attention_mask']

## 7. Initialize C2S Encoder and Classification Head

In [ ]:
# Model class definitions
class ClassificationHead(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_size, num_classes)
        
    def forward(self, x):
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

class C2SClassifier(nn.Module):
    def __init__(self, encoder, classification_head, hidden_size=256):
        super().__init__()
        self.encoder = encoder
        self.classification_head = classification_head
        self.hidden_size = hidden_size
        
    def forward(self, input_ids, attention_mask):
        # Get encoder output
        encoder_output = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = encoder_output.last_hidden_state
        
        # Mean pooling over sequence, weighted by attention mask
        mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
        sum_hidden = (last_hidden * mask_expanded).sum(1)
        sum_mask = mask_expanded.sum(1)
        pooled = sum_hidden / sum_mask.clamp(min=1e-9)
        
        # Classification head
        logits = self.classification_head(pooled)
        return logits

# Load encoder and create model with memory optimization
logger.info(f"Loading C2S encoder from {CONFIG['model_name']}")
import gc
gc.collect()  # Clear memory before loading large model
if device.type == 'cuda':
    torch.cuda.empty_cache()

# Load with memory-efficient settings
encoder = AutoModel.from_pretrained(
    CONFIG['model_name'],
    torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32,  # Use float16 on GPU to save memory
    device_map=device if device.type == 'cuda' else None,  # Auto device mapping
    low_cpu_mem_usage=True  # Load directly to GPU if available
)

# Set encoder to eval mode by default (we'll enable training later)
encoder.eval()

hidden_size = encoder.config.hidden_size
logger.info(f"Encoder hidden size: {hidden_size}")

classification_head = ClassificationHead(hidden_size, 128, num_classes, dropout=0.2)  # Reduced hidden size
model = C2SClassifier(encoder, classification_head)

logger.info(f"Model created with {sum(p.numel() for p in model.parameters()):,} parameters")
logger.info(f"Classification head: {hidden_size} -> 128 -> {num_classes}")

# Log memory usage
if device.type == 'cuda':
    torch.cuda.empty_cache()
    logger.info(f"GPU memory after model loading: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 8. Build Reachability Matrix from Ontology

In [ ]:
# Build reachability matrix
def build_reachability_matrix_local(ontology, le):
    """Build reachability matrix from ontology"""
    num_classes = len(le.classes_)
    reachability = torch.zeros((num_classes, num_classes))
    
    # Each class can reach itself
    for i in range(num_classes):
        reachability[i, i] = 1.0
    
    # Each class can reach its ancestors
    for cell_type in ontology:
        if cell_type not in le.classes_:
            continue
        child_idx = le.transform([cell_type])[0]
        
        # Walk up to ancestors
        current = ontology[cell_type].get('parent')
        while current is not None:
            if current in le.classes_:
                parent_idx = le.transform([current])[0]
                reachability[child_idx, parent_idx] = 1.0
            current = ontology.get(current, {}).get('parent')
    
    return reachability

reachability_matrix = build_reachability_matrix_local(ontology, le)
reachability_matrix = reachability_matrix.to(device)
logger.info(f"Reachability matrix shape: {reachability_matrix.shape}")
logger.info(f"Average reachability per class: {reachability_matrix.sum(1).mean():.2f}")

## 9. Configure HCE Loss and Optimizer

In [ ]:
# HCE Loss (vectorized)
class HCELoss(nn.Module):
    def __init__(self, reachability_matrix, device='cuda'):
        super().__init__()
        self.reachability_matrix = reachability_matrix.to(device)
        self.device = device
        
    def forward(self, logits, labels):
        batch_size = logits.shape[0]
        num_classes = logits.shape[1]
        
        # Softmax to get probabilities
        probs = F.softmax(logits, dim=1)
        
        # Get reachability for each label
        label_reachability = self.reachability_matrix[labels]  # [batch_size, num_classes]
        
        # HCE loss: -log(sum of probs for reachable classes)
        # Vectorized: use probs * reachability_matrix[labels] and sum
        reachable_probs = (probs * label_reachability).sum(dim=1)  # [batch_size]
        reachable_probs = torch.clamp(reachable_probs, min=1e-9)
        hce_loss = -torch.log(reachable_probs)
        
        return hce_loss.mean()

hce_loss = HCELoss(reachability_matrix, device)
logger.info("HCE Loss initialized (vectorized)")

# Optimizer with warmup and scheduling
optimizer = AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=0.01)
total_steps = len(train_idx) // CONFIG['batch_size'] * CONFIG['num_epochs']
warmup_steps = min(500, total_steps // 10)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

cosine_scheduler = CosineAnnealingLR(
    optimizer,
    T_max=total_steps - warmup_steps,
    eta_min=1e-6
)

logger.info(f"Optimizer: AdamW, lr={CONFIG['learning_rate']}, weight_decay=0.01")
logger.info(f"Schedulers: Linear warmup ({warmup_steps} steps) + Cosine annealing")

## 10. Training Loop with Checkpointing and Early Stopping

In [ ]:
# Helper functions for training
def create_dataloaders(X_indices, y_labels, batch_size, shuffle=True):
    """Create DataLoader for train/val/test"""
    class TextDataset(Dataset):
        def __init__(self, input_ids, attention_mask, labels):
            self.input_ids = input_ids
            self.attention_mask = attention_mask
            self.labels = labels
            
        def __len__(self):
            return len(self.labels)
        
        def __getitem__(self, idx):
            return {
                'input_ids': self.input_ids[idx],
                'attention_mask': self.attention_mask[idx],
                'labels': self.labels[idx]
            }
    
    dataset = TextDataset(X_indices[0], X_indices[1], y_labels)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=0)

def train_epoch(model, train_loader, hce_loss, optimizer, scheduler, device, epoch):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    batch_count = 0
    
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        logits = model(input_ids, attention_mask)
        loss = hce_loss(logits, labels)
        
        # Check for NaN/Inf
        if torch.isnan(loss) or torch.isinf(loss):
            logger.warning(f"Detected NaN/Inf loss at epoch {epoch}, batch {batch_count}")
            continue
        
        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        batch_count += 1
    
    return total_loss / max(batch_count, 1)

def evaluate(model, val_loader, hce_loss, device, le):
    """Evaluate on validation/test set"""
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            logits = model(input_ids, attention_mask)
            loss = hce_loss(logits, labels)
            
            total_loss += loss.item()
            all_preds.extend(logits.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / max(len(all_labels), 1)
    accuracy = (np.array(all_preds) == np.array(all_labels)).mean()
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    
    return avg_loss, accuracy, f1, all_preds, all_labels

# Create dataloaders
logger.info("Creating data loaders")
train_loader = create_dataloaders(
    (all_input_ids[train_idx], all_attention_masks[train_idx]),
    labels_encoded[train_idx],
    CONFIG['batch_size'],
    shuffle=True
)

val_loader = create_dataloaders(
    (all_input_ids[val_idx], all_attention_masks[val_idx]),
    labels_encoded[val_idx],
    CONFIG['batch_size'],
    shuffle=False
)

logger.info(f"Train loader: {len(train_loader)} batches")
logger.info(f"Val loader: {len(val_loader)} batches")

# Move model to device
model = model.to(device)
logger.info(f"Model moved to {device}")

# Training loop with early stopping
logger.info("Starting training loop")
best_f1 = 0
patience_counter = 0
train_losses = []
val_losses = []
val_f1_scores = []

checkpoint_dir = CONFIG['output_dir']
os.makedirs(checkpoint_dir, exist_ok=True)

for epoch in range(CONFIG['num_epochs']):
    train_loss = train_epoch(model, train_loader, hce_loss, optimizer, scheduler, device, epoch)
    val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, hce_loss, device, le)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_f1_scores.append(val_f1)
    
    logger.info(f"Epoch {epoch+1}/{CONFIG['num_epochs']}: "
                f"train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, "
                f"val_acc={val_acc:.4f}, val_f1={val_f1:.4f}")
    
    # Early stopping check
    if val_f1 > best_f1:
        best_f1 = val_f1
        patience_counter = 0
        # Save best model
        torch.save(model.state_dict(), os.path.join(checkpoint_dir, 'best_model.pth'))
        logger.info(f"New best F1: {best_f1:.4f}, saved checkpoint")
    else:
        patience_counter += 1
        
    # Save last model
    torch.save(model.state_dict(), os.path.join(checkpoint_dir, 'last_model.pth'))
    
    if patience_counter >= CONFIG['early_stopping_patience']:
        logger.info(f"Early stopping triggered at epoch {epoch+1}")
        break

logger.info(f"Training complete. Best F1: {best_f1:.4f}")

## 11. Evaluate with Flat and Hierarchical Metrics

In [ ]:
# Load best model
model.load_state_dict(torch.load(os.path.join(checkpoint_dir, 'best_model.pth')))
logger.info("Loaded best model for final evaluation")

# Create test loader
test_loader = create_dataloaders(
    (all_input_ids[test_idx], all_attention_masks[test_idx]),
    labels_encoded[test_idx],
    CONFIG['batch_size'],
    shuffle=False
)

# Evaluate on test set
test_loss, test_acc, test_f1, test_preds, test_labels = evaluate(model, test_loader, hce_loss, device, le)

logger.info("\n" + "="*50)
logger.info("FINAL TEST RESULTS")
logger.info("="*50)
logger.info(f"Test Loss: {test_loss:.4f}")
logger.info(f"Test Accuracy: {test_acc:.4f}")
logger.info(f"Test F1 (weighted): {test_f1:.4f}")

# Per-class metrics
report = classification_report(test_labels, test_preds, target_names=le.classes_, zero_division=0)
logger.info(f"\nPer-class metrics:\n{report}")

# Hierarchical accuracy (ancestor credit)
def compute_hierarchical_accuracy(preds, labels, reachability_matrix):
    """Compute accuracy giving partial credit for ancestor predictions"""
    correct = 0
    for pred, label in zip(preds, labels):
        if reachability_matrix[label, pred] > 0:
            correct += 1
    return correct / len(labels)

hierarchical_acc = compute_hierarchical_accuracy(
    np.array(test_preds),
    np.array(test_labels),
    reachability_matrix.cpu()
)
logger.info(f"Hierarchical Accuracy (ancestor-credit): {hierarchical_acc:.4f}")

# Save results
results_dict = {
    'test_loss': float(test_loss),
    'test_accuracy': float(test_acc),
    'test_f1': float(test_f1),
    'hierarchical_accuracy': float(hierarchical_acc),
    'train_losses': train_losses,
    'val_losses': val_losses,
    'val_f1_scores': val_f1_scores
}

with open(os.path.join(checkpoint_dir, 'results.json'), 'w') as f:
    json.dump(results_dict, f, indent=2)

logger.info(f"Results saved to {os.path.join(checkpoint_dir, 'results.json')}")

## 12. Generate Visualizations and Persist All Artifacts

In [ ]:
# Visualize training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss curves
axes[0].plot(train_losses, label='Train Loss', marker='o')
axes[0].plot(val_losses, label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# F1 curves
axes[1].plot(val_f1_scores, label='Val F1', marker='o', color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1 Score')
axes[1].set_title('Validation F1 Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(checkpoint_dir, 'training_curves.png'), dpi=150)
logger.info(f"Training curves saved to {os.path.join(checkpoint_dir, 'training_curves.png')}")
plt.close()

# Confusion matrix
cm = confusion_matrix(test_labels, test_preds)
fig, ax = plt.subplots(figsize=(12, 10))
ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(ax=ax, cmap='Blues')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(checkpoint_dir, 'confusion_matrix.png'), dpi=150)
logger.info(f"Confusion matrix saved to {os.path.join(checkpoint_dir, 'confusion_matrix.png')}")
plt.close()

# Save predictions and embeddings
predictions_df = pd.DataFrame({
    'predicted_class': [le.classes_[p] for p in test_preds],
    'true_class': [le.classes_[l] for l in test_labels],
    'correct': np.array(test_preds) == np.array(test_labels)
})
predictions_df.to_csv(os.path.join(checkpoint_dir, 'test_predictions.csv'), index=False)
logger.info(f"Predictions saved to {os.path.join(checkpoint_dir, 'test_predictions.csv')}")

# Save ontology and configuration
with open(os.path.join(checkpoint_dir, 'ontology.json'), 'w') as f:
    json.dump(ontology, f, indent=2)

logger.info(f"Ontology saved to {os.path.join(checkpoint_dir, 'ontology.json')}")

# Final summary
summary = f"""
===========================================
INTEGRATED HCE TRAINING - FINAL SUMMARY
===========================================

Dataset Information:
- Total cells after balancing: {len(adata)}
- Number of cell types: {num_classes}
- Train/Val/Test split: {len(train_idx)}/{len(val_idx)}/{len(test_idx)}

Model Configuration:
- Encoder: {CONFIG['model_name']}
- Hidden size: {hidden_size}
- Classification head: {hidden_size} -> 256 -> {num_classes}
- Total parameters: {sum(p.numel() for p in model.parameters()):,}

Training Details:
- Optimizer: AdamW (lr={CONFIG['learning_rate']}, weight_decay=0.01)
- Loss function: Vectorized HCE
- Batch size: {CONFIG['batch_size']}
- Epochs trained: {len(train_losses)}
- Best validation F1: {best_f1:.4f}

Final Test Results:
- Test Accuracy: {test_acc:.4f}
- Test F1 (weighted): {test_f1:.4f}
- Hierarchical Accuracy: {hierarchical_acc:.4f}

Artifacts:
- Best model: {os.path.join(checkpoint_dir, 'best_model.pth')}
- Last model: {os.path.join(checkpoint_dir, 'last_model.pth')}
- Results: {os.path.join(checkpoint_dir, 'results.json')}
- Predictions: {os.path.join(checkpoint_dir, 'test_predictions.csv')}
- Config: {os.path.join(checkpoint_dir, 'config.json')}
- Ontology: {os.path.join(checkpoint_dir, 'ontology.json')}
- Training curves: {os.path.join(checkpoint_dir, 'training_curves.png')}
- Confusion matrix: {os.path.join(checkpoint_dir, 'confusion_matrix.png')}

===========================================
"""

logger.info(summary)

with open(os.path.join(checkpoint_dir, 'training_summary.txt'), 'w') as f:
    f.write(summary)

logger.info("Training complete! All artifacts saved.")